# 第 2 章 三面等価と産業連関表: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/02-three-way-equivalence.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
output = [100, 300, 500]        # 各産業の産出額
intermediate = [0, 100, 300]    # 各産業の中間投入
compensation = 280              # 雇用者報酬
operating_surplus = 120         # 営業余剰
depreciation = 80               # 固定資本減耗
indirect_taxes = 20             # 間接税（補助金は 0 とする）
consumption = 290               # 民間消費
government = 80                 # 政府消費
investment = 100                # 総固定資本形成
inventory = 10                  # 在庫品増加
net_exports = 20                # 純輸出
coefficients = [[0.2, 0.3], [0.4, 0.1]]  # 投入係数 A（行: 売り手、列: 買い手）
final_demand = [100, 200]       # 最終需要

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 三面等価

In [ ]:
from lib.chapter02.three_way import (
    gdp_distribution,
    gdp_expenditure,
    gdp_production,
    is_three_way_equal,
)

production = gdp_production(output, intermediate)
distribution = gdp_distribution(
    compensation, operating_surplus, depreciation, indirect_taxes, 0
)
expenditure = gdp_expenditure(
    consumption, government, investment, inventory, net_exports
)
print(f"生産面 {production}、分配面 {distribution}、支出面 {expenditure}")
equal = is_three_way_equal(production, distribution, expenditure)
print("三面等価が成り立つか:", equal)

## 図: 三面等価

In [ ]:
from lib.chapter02.plot import plot_three_way

plot_three_way(
    {
        "Production": {
            f"Industry {i + 1}": o - m
            for i, (o, m) in enumerate(zip(output, intermediate, strict=True))
        },
        "Distribution": {
            "Compensation": compensation,
            "Operating surplus": operating_surplus,
            "Depreciation": depreciation,
            "Taxes less subsidies": indirect_taxes,
        },
        "Expenditure": {
            "Consumption": consumption,
            "Government": government,
            "Investment": investment,
            "Inventory": inventory,
            "Net exports": net_exports,
        },
    }
)

## 産業連関表: 最終需要を満たす生産量

In [ ]:
from lib.chapter02.input_output import leontief_inverse, output_for_demand

print("レオンチェフ逆行列:")
print(leontief_inverse(coefficients).round(3))
print("必要な生産量:", output_for_demand(coefficients, final_demand).round(1))

In [ ]:
from lib.chapter02.plot import plot_io_output

plot_io_output(coefficients, final_demand, ["Manufacturing", "Services"])

## 試してみよう

- `inventory = 30` にすると、支出面は 520 になり、生産面・分配面（500）と
  ずれて、「三面等価が成り立つか」は `False` になります。売れ残り（在庫）は、
  支出に数えるので、帳尻は在庫で合います
- `final_demand = [200, 200]` にすると、必要な生産量は [400.0, 400.0] になります
  （[100, 200] のときは [250.0, 333.3]）。最終需要の増加は、中間投入を通じて、
  それ以上の生産を呼びます
- `coefficients = [[0.4, 0.3], [0.4, 0.1]]` にすると、必要な生産量は
  [357.1, 381.0] に増えます。投入係数が大きい（原材料を多く使う）ほど、生産が膨らみます